In [1]:
using LinearAlgebra
using LinearOperators
using SparseArrays

In [2]:
function get_1d_laplace_op_matrix(n)
	off = ones(n-1)
	diag = ones(n)
	spdiagm(-1 => off, 0 => -2diag, 1 => off)
end

get_1d_laplace_op_matrix (generic function with 1 method)

In [3]:
# L
# E L, L E
# E E L, E L E, L E E
# E E E L, E E L E, E L E E, L E E E
# (E E E L), (E E L) (E), (E L) (E E), (L) (E E E)
# E ... E L = L of size n^(k+1) if E is there k times

# (E E E L), (E E L) (E), (E L) (E E), (L) (E E E)

# E E E x L
# (E E x L) x (E)
# (E x L) x (E E)
# L x E E E

function get_laplace_op_matrix(n, d)
	L = get_1d_laplace_op_matrix(n)
	if d == 1
		L_full = L
	elseif d == 2
		E = sparse(I, n, n)
		L_full = kron(E, L) + kron(L, E)
	elseif d >= 3
		# step 1
		size = n^(d-1)
		E = sparse(I, size, size)
		L_full = kron(E, L)
		# step 2:d-1
		for k in 2:(d-1)
			n_Es_lhs = d-k
			n_Es_rhs = k-1
			# lhs
			size = n^(n_Es_lhs)
			E = sparse(I, size, size)
			kron_lhs = kron(E, L)
			# rhs
			size = n^(n_Es_rhs)
			E = sparse(I, size, size)
			L_full += kron(kron_lhs, E)
		end
		# step d
		size = n^(d-1)
		E = sparse(I, size, size)
		L_full += kron(L, E)
	end
end

get_laplace_op_matrix (generic function with 1 method)

In [4]:
get_laplace_op_matrix(3, 2)

9×9 SparseMatrixCSC{Float64, Int64} with 33 stored entries:
 -4.0   1.0    ⋅    1.0    ⋅     ⋅     ⋅     ⋅     ⋅ 
  1.0  -4.0   1.0    ⋅    1.0    ⋅     ⋅     ⋅     ⋅ 
   ⋅    1.0  -4.0    ⋅     ⋅    1.0    ⋅     ⋅     ⋅ 
  1.0    ⋅     ⋅   -4.0   1.0    ⋅    1.0    ⋅     ⋅ 
   ⋅    1.0    ⋅    1.0  -4.0   1.0    ⋅    1.0    ⋅ 
   ⋅     ⋅    1.0    ⋅    1.0  -4.0    ⋅     ⋅    1.0
   ⋅     ⋅     ⋅    1.0    ⋅     ⋅   -4.0   1.0    ⋅ 
   ⋅     ⋅     ⋅     ⋅    1.0    ⋅    1.0  -4.0   1.0
   ⋅     ⋅     ⋅     ⋅     ⋅    1.0    ⋅    1.0  -4.0

## define RHS and solve using different methods

$u_t = \alpha\, \Delta u \:\:\text{on}\:\:\Omega$

initial & boundary conditions:

$u(t=0, x) = u_0(x) \:\:\text{on}\:\:\Omega$

$u(t, x) = u_{D}(t,x) \:\:\text{on}\:\:\partial\Omega$

consider the discretization in 1 dimension:

$\partial u / \partial x \approx \frac{u(x+h/2) - u(x-h/2)}{h}$

$\partial^2 u /\partial x^2 \approx \frac{u(x+h) - 2u(x) + u(x-h)}{h^2}
\approx \frac{u_{i+1} - 2u_i + u_{i-1}}{h^2}$

$\partial u / \partial t \approx \frac{u(x+\tau) - u(x)}{\tau}
\approx \frac{u^{t+1} - u^{t}}{\tau}$

### explicit

$
\frac{u_i^{t+1} - u_i^{t}}{\tau}
= \alpha\, \frac{u_{i+1}^{t} - 2u_i^{t} + u_{i-1}^{t}}{h^2}
$

$
u_i^{t+1}
= \frac{\alpha\,\tau}{h^2} (u_{i+1}^{t} - 2u_i^{t} + u_{i-1}^{t}) + u_i^{t}
$

$
U^{t+1}
= \frac{\alpha\,\tau}{h^2} L\,U^t + U^{t}
= (I + \frac{\alpha\,\tau}{h^2}L) \, U^{t}
$

We just iteratively compute $U^{t+1}$ according to:

$U^{t+1} = (I + \frac{\alpha\,\tau}{h^2}L) \, U^{t}$

where $U_0$ is given by the initial condition.

### implicit

$
\frac{u_i^{t+1} - u_i^{t}}{\tau}
= \alpha\, \frac{u_{i+1}^{t+1} - 2u_i^{t+1} + u_{i-1}^{t+1}}{h^2}
$

$
u_i^{t+1} -
\frac{\alpha\,\tau}{h^2} (u_{i+1}^{t+1} - 2u_i^{t+1} + u_{i-1}^{t+1})
= u_i^{t}
$

$
U^{t+1} -
\frac{\alpha\,\tau}{h^2} L\,U^{t+1}
= (I - \frac{\alpha\,\tau}{h^2} L)\, U^{t+1}
= U^t
$

To get $U^{t+1}$ from $U^{t}$, we need to solve:

$(I - \frac{\alpha\,\tau}{h^2} L)\, U^{t+1} = U^t$

(We need to solve $A\,x=b$, where $x$ and $b$ are of size $n^d$.)

In [5]:
alpha = 2

2

In [6]:
n = 50
d = 1
N = n^d

50

In [7]:
h = 1 / (n+1)

0.0196078431372549

In [8]:
n_iters = 10
tau = 1 / (n_iters + 1)

0.09090909090909091

In [9]:
function u_analytic_fun(t, x)
    prod(sin.(pi*x)) * exp(-alpha * t)

    #prod(x)*prod(1 .- x)
end

u_analytic_fun (generic function with 1 method)

### here we go

In [10]:
function get_grid_points_as_1d_vect(n, d)
    a = 0
    b = 1
    h = 1/(n+1)
    xs = [range(h, 1-h; length=n) for _ in 1:d]
    coords = collect(Iterators.product(xs...))
    [collect(x) for x in coords[1:end]]
end

get_grid_points_as_1d_vect (generic function with 1 method)

In [11]:
n^d

50

In [12]:
grid_points_as_1d_vect = get_grid_points_as_1d_vect(n,d);

In [13]:
#U_analytic = u_analytic_fun.(0, grid_points_as_1d_vect);

In [14]:
U_0 = u_analytic_fun.(0, grid_points_as_1d_vect);

In [15]:
L = get_laplace_op_matrix(n,d);

In [16]:
r = alpha * tau / h^2

472.90909090909093

In [17]:
using IterativeSolvers

### Explicit

In [18]:
A_expl = sparse(I, N, N) + r * L;

In [19]:
#U_t = A_expl * U_0;

In [20]:
U_evol_expl = zeros(n_iters, N)
U_evol_expl[1, :] = U_0
for t in 2:1:n_iters
    U_evol_expl[t, :] = A_expl * U_evol_expl[t-1, :];
end

In [21]:
U_evol_expl

10×50 Matrix{Float64}:
  0.0615609      0.122888      0.18375     …    0.122888     0.0615609
 -0.0488736     -0.0975618    -0.14588         -0.0975618   -0.0488736
  0.038801       0.0774549     0.115815         0.0774549    0.038801
 -0.0308044     -0.0614919    -0.0919462       -0.0614919   -0.0308046
  0.0244631      0.0487825     0.07309          0.0488959    0.0246945
 -0.043474       0.0431772    -0.245903    …   -0.3382      -0.208525
 61.4939      -177.643       373.833          808.955       37.0804
 -1.4211e5       3.73711e5    -7.32794e5       -1.90142e6    3.47528e5
  3.10999e8     -7.66838e8     1.41831e9        4.15481e9   -1.22755e9
 -6.56482e11     1.54233e12   -2.71585e12      -8.68535e12   3.12466e12

### Implicit

In [22]:
A_impl = sparse(I, N, N) - r * L;

In [23]:
U_evol_impl = zeros(n_iters, N)
U_evol_impl[1, :] = U_0
for t in 2:1:n_iters
    U_evol_impl[t, :] = cg(A_impl, U_evol_impl[t-1, :]; verbose=true);
end

  1	3.00e-13

  1	1.31e-13

  1	4.62e-14

  1	1.71e-14

  1	6.18e-15

  1	2.26e-15

  1	8.58e-16

  1	3.61e-16

  1	1.06e-16



In [24]:
U_evol_impl

10×50 Matrix{Float64}:
 0.0615609    0.122888     0.18375      …  0.122888     0.0615609
 0.022034     0.0439844    0.065768        0.0439844    0.022034
 0.00788645   0.015743     0.0235398       0.015743     0.00788645
 0.00282273   0.00563476   0.0084254       0.00563476   0.00282273
 0.00101032   0.0020168    0.00301564      0.0020168    0.00101032
 0.000361615  0.000721857  0.00107936   …  0.000721857  0.000361615
 0.00012943   0.000258369  0.000386327     0.000258369  0.00012943
 4.63257e-5   9.24757e-5   0.000138275     9.24757e-5   4.63257e-5
 1.6581e-5    3.30991e-5   4.94916e-5      3.30991e-5   1.6581e-5
 5.9347e-6    1.18469e-5   1.77141e-5      1.18469e-5   5.9347e-6

### operator approach

In [25]:
function laplace_operator!(v_new, v)

    # from vector of value to values on a grid
    U = reshape(v, ntuple(i -> n, d)) 

    # apply the laplace stencil on every value on the grid
    LU = zeros(size(U));
    for dim in 1:d
        index_rs = ntuple(i -> i == dim ? (2:n) : Colon(), d)
        index_ls = ntuple(i -> i == dim ? (1:(n-1)) : Colon(), d)
        LU[index_rs...] .+= U[index_ls...]
        LU[index_ls...] .+= U[index_rs...]
    end
    LU .-= 2*d*U

    v_new .= vec(LU)
end


laplace_operator! (generic function with 1 method)

In [26]:
#L_op = LinearOperator(Float64, N, N, true, true, laplace_operator!)

In [27]:
#U_cg_op = cg(L_op, F; maxiter=100, verbose=true)

In [28]:
d

1

In [29]:
if d == 1
    using Plots

    # Create a grid
    t = tau:tau:(1-tau)
    x = h:h:(1-h)
    #z = U_evol_impl
    #z = [U_cg_op[i*n+j] for i in 0:(n-1), j in 1:n]
    #z = [U_analytic[i*n+j] for i in 0:(n-1), j in 1:n]
    z = [U_evol_impl[i, :] for i in 1:n_iters]

    # Make the surface plot
    surface(t, x, z, xlabel="t", ylabel="x", zlabel="z")
end

In [30]:
surface(t, x, z, xlabel="t", ylabel="x", zlabel="z")

In [31]:
z

10-element Vector{Vector{Float64}}:
 [0.061560906133942835, 0.12288829066471411, 0.18374951781657034, 0.24391372010837714, 0.30315267411304353, 0.3612416661871529, 0.41796034488678346, 0.47309355683601007, 0.5264321628773557, 0.5777738314082511  …  0.5777738314082512, 0.5264321628773561, 0.4730935568360101, 0.41796034488678324, 0.3612416661871533, 0.30315267411304364, 0.24391372010837745, 0.18374951781657037, 0.12288829066471434, 0.06156090613394323]
 [0.022033992012878204, 0.04398440154683836, 0.06576796317993411, 0.08730204440145681, 0.10850495906535243, 0.1292962772537531, 0.14959713037520506, 0.16933051034025032, 0.18842156167948756, 0.20679786549600734  …  0.20679786549600737, 0.18842156167948768, 0.16933051034025035, 0.14959713037520497, 0.12929627725375323, 0.10850495906535247, 0.08730204440145692, 0.06576796317993412, 0.043984401546838435, 0.022033992012878346]
 [0.007886446683667177, 0.015742977373750865, 0.023539789558278235, 0.03124730725802317, 0.03883629321832602, 0.046277

In [32]:
U_evol_impl

10×50 Matrix{Float64}:
 0.0615609    0.122888     0.18375      …  0.122888     0.0615609
 0.022034     0.0439844    0.065768        0.0439844    0.022034
 0.00788645   0.015743     0.0235398       0.015743     0.00788645
 0.00282273   0.00563476   0.0084254       0.00563476   0.00282273
 0.00101032   0.0020168    0.00301564      0.0020168    0.00101032
 0.000361615  0.000721857  0.00107936   …  0.000721857  0.000361615
 0.00012943   0.000258369  0.000386327     0.000258369  0.00012943
 4.63257e-5   9.24757e-5   0.000138275     9.24757e-5   4.63257e-5
 1.6581e-5    3.30991e-5   4.94916e-5      3.30991e-5   1.6581e-5
 5.9347e-6    1.18469e-5   1.77141e-5      1.18469e-5   5.9347e-6